In [0]:
company_lookup = spark.table('plstocks.bronze_company_name_lookup')
# company_lookup.display()

In [0]:
sector_lookup = spark.table('plstocks.silver_sector_lookup')
# sector_lookup.display()

In [0]:
company_insights = company_lookup.join(sector_lookup, 'ticker', 'left').drop('insert_timestamp')
# display(company_insights)

In [0]:
cashflow = spark.table('plstocks.silver_cashflow')
# display(cashflow)

In [0]:
from pyspark.sql.functions import max, col

cashflow_agg = cashflow.groupBy('ticker').agg(max('date').alias('date'))
cashflow_result = cashflow.join(cashflow_agg, on=['ticker', 'date'])
cashflow_to_join = cashflow_result.select(col('ticker'), col('free_cashflow'))
# display(cashflow_to_join)

In [0]:
company_insights = company_insights.join(cashflow_to_join, 'ticker', 'left').withColumnRenamed('free_cashflow', 'latest_free_cashflow')
# display(company_insights)

In [0]:
stock_prices = spark.table('plstocks.silver_stocks_price')
# display(stock_prices)

In [0]:
from pyspark.sql.functions import max, col
from pyspark.sql.types import DoubleType

stocks_agg = stock_prices.groupBy('TICKER').agg(max('date').alias('date'))
stocks_result = stock_prices.join(stocks_agg, on=['ticker', 'date'])
stocks_to_join = stocks_result.select(col('TICKER').alias('ticker'), col('CLOSE').cast(DoubleType()), col('ACTIVE'))
# display(stocks_to_join)

In [0]:
company_insights = company_insights.join(stocks_to_join, 'ticker', 'left').withColumnRenamed('CLOSE', 'latest_stock_price').withColumnRenamed('ACTIVE', 'active_company')
# display(company_insights)

In [0]:
financial_reports = spark.table('plstocks.silver_financial_reports')
# display(financial_reports)

In [0]:
from pyspark.sql.functions import max, col

reports_agg = financial_reports.groupBy('ticker').agg(max('year').alias('year'), max('insert_timestamp').alias('insert_timestamp'))
reports_result = financial_reports.join(reports_agg, on=['ticker', 'year', 'insert_timestamp'])
reports_to_join = reports_result.select(col('ticker'), col('sales_revenue'))
# display(reports_to_join)

In [0]:
company_insights = company_insights.join(reports_to_join, 'ticker', 'left')
# display(company_insights)

In [0]:
financial_ratios = spark.table('plstocks.silver_financial_ratios')
# display(financial_ratios)

In [0]:
from pyspark.sql.functions import max, first

ratios_agg = financial_ratios.groupBy('ticker').agg(max('insert_timestamp').alias('insert_timestamp'), first('price_sales').alias('price_sales'))
ratios_to_join = financial_ratios.join(ratios_agg, on=['ticker', 'insert_timestamp', 'price_sales']).groupBy('ticker').agg(first('price_sales').alias('price_sales'))
# display(ratios_to_join)

In [0]:
company_insights = company_insights.join(ratios_to_join, 'ticker', 'left')
# display(company_insights)

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType

company_insights = company_insights.withColumn('shares_volume', (col('price_sales') * (col('sales_revenue') / col('latest_stock_price'))).cast(IntegerType()))
# display(company_insights)

In [0]:
from pyspark.sql.functions import col
from pyspark.sql.types import FloatType

company_insights = company_insights.withColumn('market_capitalization', (col('latest_stock_price') * col('shares_volume')).cast(FloatType()))
company_insights = company_insights.drop(col('sales_revenue'), col('price_sales'))
# display(company_insights)

In [0]:
dividends = spark.table('plstocks.silver_dividend_history')
# display(dividends)

In [0]:
from pyspark.sql.functions import col, current_timestamp, year

dividends_agg = dividends.groupBy('ticker').agg(max('dividend_year').alias('dividend_year'), max('insert_timestamp').alias('insert_timestamp'))
dividends_result = dividends.join(dividends_agg, on=['ticker', 'dividend_year', 'insert_timestamp'])
dividends_latest = dividends_result.select(col('ticker'), col('dividend_year'), col('dividend_per_share'))
dividends_to_join = dividends_latest.filter(col('dividend_year') >= year(current_timestamp())-2).drop(col('dividend_year'))
# display(dividends_to_join)

In [0]:
company_insights = company_insights.join(dividends_to_join, 'ticker', 'left')
# display(company_insights)

In [0]:
from pyspark.sql.functions import col, round

company_insights = company_insights.withColumn('dividend_yield', (col('dividend_per_share') * 100 / col('latest_stock_price')))
company_insights = company_insights.withColumn('dividend_yield', round(col('dividend_yield'), 2))
# display(company_insights)

In [0]:
company_insights.write.mode("overwrite").saveAsTable("plstocks.gold_company_insights")